# Homework 6

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import Dash, html, dash_table, dcc
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc


I will be using the dataset that was given, since I tired looking online (Harvard Dataverse) and couldn't find one that I would like. 

In [4]:
data = pd.read_csv('student-por.csv')
data.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [5]:
data.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')

# Attributes for both student-mat.csv (Math course) and student-por.csv (Portuguese language course) datasets:
1 school - student's school (binary: "GP" - Gabriel Pereira or "MS" - Mousinho da Silveira)

2 sex - student's sex (binary: "F" - female or "M" - male)

3 age - student's age (numeric: from 15 to 22)

4 address - student's home address type (binary: "U" - urban or "R" - rural)

5 famsize - family size (binary: "LE3" - less or equal to 3 or "GT3" - greater than 3)

6 Pstatus - parent's cohabitation status (binary: "T" - living together or "A" - apart)

7 Medu - mother's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)

8 Fedu - father's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)

9 Mjob - mother's job (nominal: "teacher", "health" care related, civil "services" (e.g. administrative or police), "at_home" or "other")

10 Fjob - father's job (nominal: "teacher", "health" care related, civil "services" (e.g. administrative or police), "at_home" or "other")

11 reason - reason to choose this school (nominal: close to "home", school "reputation", "course" preference or "other")

12 guardian - student's guardian (nominal: "mother", "father" or "other")

13 traveltime - home to school travel time (numeric: 1 - <15 min., 2 - 15 to 30 min., 3 - 30 min. to 1 hour, or 4 - >1 hour)

14 studytime - weekly study time (numeric: 1 - <2 hours, 2 - 2 to 5 hours, 3 - 5 to 10 hours, or 4 - >10 hours)

15 failures - number of past class failures (numeric: n if 1<=n<3, else 4)

16 schoolsup - extra educational support (binary: yes or no)

17 famsup - family educational support (binary: yes or no)

18 paid - extra paid classes within the course subject (Math or Portuguese) (binary: yes or no)

19 activities - extra-curricular activities (binary: yes or no)

20 nursery - attended nursery school (binary: yes or no)

21 higher - wants to take higher education (binary: yes or no)

22 internet - Internet access at home (binary: yes or no)

23 romantic - with a romantic relationship (binary: yes or no)

24 famrel - quality of family relationships (numeric: from 1 - very bad to 5 - excellent)

25 freetime - free time after school (numeric: from 1 - very low to 5 - very high)

26 goout - going out with friends (numeric: from 1 - very low to 5 - very high)

27 Dalc - workday alcohol consumption (numeric: from 1 - very low to 5 - very high)

28 Walc - weekend alcohol consumption (numeric: from 1 - very low to 5 - very high)

29 health - current health status (numeric: from 1 - very bad to 5 - very good)

30 absences - number of school absences (numeric: from 0 to 93)


# these grades are related with the course subject, Math or Portuguese:

31 G1 - first period grade (numeric: from 0 to 20)

31 G2 - second period grade (numeric: from 0 to 20)

32 G3 - final grade (numeric: from 0 to 20, output target)

Additional note: there are several (382) students that belong to both datasets . 
These students can be identified by searching for identical attributes
that characterize each student, as shown in the annexed R file.


### Columns to compare:

Before I do anything in terms of building our dashboard, I'll figure out what data I will use to represent and get that data ready to be shown.

data 1: We will compare weekly study time (studytime) to the final grade (G3) , and group by extra educational support (schoolsup | binary yes no)

data 1: boxplot

data 2: We will compare the number of school absences (absences) to the final grade (G3), and group it by the students current health status (health)

data 2: scatter plot

data 3: We will compare quality of family relationships (famrel) to how they did in their final grade (G3), grouped by if they had extra family educational support (famsup | binary yes or no)

data 3: bar plot

data 4: We will compare weekend alcohol consumption (Walc) to the number of times going out with friends (goout), grouped by the number of past class failures (failures | numeric 1-4)

data 4: heatmap

In [14]:
grades = data[['G1', 'G2', 'G3']].columns

lowest = data['absences'].min()
highest = data['absences'].max()

In [26]:
app = dash.Dash(__name__, external_stylesheets = [dbc.themes.BOOTSTRAP])



app.layout = dbc.Container(
    [
        dbc.Row(
                dbc.Col(
                        html.Div(
                            [
                                html.H2('Student School Metrics', className = 'label-font'),
                                html.P('Exploring different academic, health, and lifestyle metrics for students', 
                                       className = 'paragraph-text'),
                            ],
                            className = 'header-block',
                        ),
                        width = 12,
                ),
                className = 'mb-3',
        ),
        
        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 1 menu', className = 'label-font'),
                            dcc.Dropdown(
                                id = 'Graph1_grade',
                                options = grades,
                                value = 'G3',
                                clearable = False,
                                className = 'dropdown-background',
                            ),

                            dbc.Switch(
                                id = 'Graph1_points',
                                label = 'On/Off',
                                value = True,
                                className = 'switch-style',
                            ),
                        ],
                        className = 'section-box',
                    ),
                    md = 4,
                    className = 'mb-4',
                ),

                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 1: Study Time vs. Grade', className = 'label-font'),
                            dcc.Graph(id = 'graph1', className = 'graph-shadow'),
                        ],
                        className = 'section-box',
                    ),
                    md = 8,
                    className = 'mb-4',
                ),
            ]
        ),
        
        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 2 menu', className = 'label-font'),

                            dcc.Dropdown(
                                id = 'Graph2_grade',
                                options = grades,
                                value = 'G3',
                                clearable = False,
                                className = 'dropdown-background',
                            ),

                            dcc.RangeSlider(
                                id = 'Graph2_Absences',
                                min = lowest,
                                max = highest,
                                step = 1,
                                value = [lowest, highest],
                                tooltip = {'placement': 'bottom'},
                                className = 'slider-style',
                            ),

                            dbc.Switch(
                                id = 'Graph2_trend',
                                label = 'Trendline',
                                value = True,
                                className = 'switch-style',
                            ),
                        ],
                        className = 'section-box',
                    ),
                    md = 4,
                    className = 'mb-4',
                ),

                 dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 2: Absences vs. Grade', className = 'label-font'),
                            
                            dcc.Graph(id = 'graph2', className = 'graph-shadow'),
                        ],
                        className = 'section-box',
                    ),
                    md = 8,
                    className = 'mb-4',
                ),
            ]
        ),

        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 3: Family Relationship vs. Grade', className = 'label-font'),
                            
                            dcc.Graph(id = 'graph3', className = 'graph-shadow'),
                        ],
                        className = 'section-box',
                    ),
                    md = 6,
                    className = 'mb-4',
                ),
                
                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 4: Weekend alc. Consumption vs Going out', className = 'label-font'),
                            dcc.Graph(id = 'graph4', className = 'graph-shadow'),
                        ],
                        className = 'section-box',
                    ),
                    md = 6,
                    className = 'mb-4',
                ),
            ]
        ),

    ],
    fluid = True,
)



@app.callback(
    Output('graph1', 'figure'),
    Output('graph2', 'figure'),
    Output('graph3', 'figure'),
    Output('graph4', 'figure'),
    Input('Graph1_grade', 'value'),
    Input('Graph1_points', 'value'),
    Input('Graph2_grade', 'value'),
    Input('Graph2_Absences', 'value'),
    Input('Graph2_trend', 'value'),
)



def update_all_graphs(graph1_grade, graph1_points, graph2_grade, absence, trend):

    fig1 = px.box(data, x = 'studytime', y = graph1_grade, color = 'schoolsup', points = graph1_points,
                  title = 'Study time vs. grade')
    fig1.update_layout(
        autosize = True)




    return fig1, fig2, fig3, fig4

if __name__ == "__main__":
    app.run(debug=True, port=8063)


## copy and paste for .py file, putting it all together in one cell.

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import Dash, html, dash_table, dcc
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc


data = pd.read_csv('student-por.csv')


grades = data[['G1', 'G2', 'G3']].columns


lowest = data['absences'].min()
highest = data['absences'].max()


app = dash.Dash(__name__, external_stylesheets = [dbc.themes.BOOTSTRAP])


app.layout = dbc.Container(
    [
        dbc.Row(
                dbc.Col(
                        html.Div(
                            [
                                html.H2('Student School Metrics', className = 'label-font'),
                                html.P('Exploring different academic, health, and lifestyle metrics for students', 
                                       className = 'paragraph-text'),
                            ],
                            className = 'header-block',
                        ),
                        width = 12,
                ),
                className = 'mb-3',
        ),
        
        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 1 menu', className = 'label-font'),
                            dcc.Dropdown(
                                id = 'Graph1_grade',
                                options = grades,
                                value = 'G3',
                                clearable = False,
                                className = 'dropdown-background',
                            ),

                            dbc.Switch(
                                id = 'Graph1_points',
                                label = 'On/Off',
                                value = True,
                                className = 'switch-style',
                            ),
                        ],
                        className = 'section-box',
                    ),
                    md = 4,
                    className = 'mb-4',
                ),

                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 1: Study Time vs. Grade', className = 'label-font'),
                            dcc.Graph(id = 'graph1', className = 'graph-shadow'),
                        ],
                        className = 'section-box',
                    ),
                    md = 8,
                    className = 'mb-4',
                ),
            ]
        ),
        
        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 2 menu', className = 'label-font'),

                            dcc.Dropdown(
                                id = 'Graph2_grade',
                                options = grades,
                                value = 'G3',
                                clearable = False,
                                className = 'dropdown-background',
                            ),

                            dcc.RangeSlider(
                                id = 'Graph2_Absences',
                                min = lowest,
                                max = highest,
                                step = 1,
                                value = [lowest, highest],
                                tooltip = {'placement': 'bottom'},
                                className = 'slider-style',
                            ),

                            dbc.Switch(
                                id = 'Graph2_trend',
                                label = 'Trendline',
                                value = True,
                                className = 'switch-style',
                            ),
                        ],
                        className = 'section-box',
                    ),
                    md = 4,
                    className = 'mb-4',
                ),

                 dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 2: Absences vs. Grade', className = 'label-font'),
                            
                            dcc.Graph(id = 'graph2', className = 'graph-shadow'),
                        ],
                        className = 'section-box',
                    ),
                    md = 8,
                    className = 'mb-4',
                ),
            ]
        ),

        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 3: Family Relationship vs. Grade', className = 'label-font'),
                            
                            dcc.Graph(id = 'graph3', className = 'graph-shadow'),
                        ],
                        className = 'section-box',
                    ),
                    md = 6,
                    className = 'mb-4',
                ),
                
                dbc.Col(
                    html.Div(
                        [
                            html.H5('Graph 4: Weekend alc. Consumption vs Going out', className = 'label-font'),
                            dcc.Graph(id = 'graph4', className = 'graph-shadow'),
                        ],
                        className = 'section-box',
                    ),
                    md = 6,
                    className = 'mb-4',
                ),
            ]
        ),

    ],
    fluid = True,
)



@app.callback(
    Output('graph1', 'figure'),
    Output('graph2', 'figure'),
    Output('graph3', 'figure'),
    Output('graph4', 'figure'),
    Input('Graph1_grade', 'value'),
    Input('Graph1_points', 'value'),
    Input('Graph2_grade', 'value'),
    Input('Graph2_Absences', 'value'),
    Input('Graph2_trend', 'value'),
)



def update_graphs(graph1_grade, graph1_points, graph2_grade, absence, trend):

    # data 1: We will compare weekly study time (studytime) to the final grade (G3) 
    # , and group by extra educational support (schoolsup | binary yes no) #boxplot

    points_setting = 'all' if graph1_points else False
    
    fig1 = px.box(data, x = 'studytime', y = graph1_grade, color = 'schoolsup', points = points_setting,
                  title = 'Study time vs. grade')
    fig1.update_layout(
        autosize = True)

    # data 2: We will compare the number of school absences (absences) to the final 
    # grade (G3), and group it by the students current health status (health) #scatterplot
    
    low, high = absence
    data2 = data[ (data['absences'] >= low) & (data['absences'] <= high) ]

    fig2 = px.scatter(data2, x = 'absences', y = graph2_grade, color = 'health', opacity = 0.7,
                      title = 'Absences vs. Grade')
    fig2.update_layout(
        autosize = True)

    # data 3: We will compare quality of family relationships (famrel) to how they did in their final grade 
    # (G3), grouped by if they had extra family educational support (famsup | binary yes or no)
    
    # since famsup is binary, we are going to take the mean finals by family relationship (numerical),
    # and group by family support

    grouped = data.groupby(['famrel', 'famsup'])['G3'].mean().reset_index()

    fig3 = px.bar(grouped, x = 'famrel', y = 'G3', color = 'famsup', barmode = 'group',
                  title = 'Mean final grade by family relationship (filtered by family support)')
    fig3.update_layout(autosize = True)

    # data 4: We will compare weekend alcohol consumption (Walc) to the number of times going out with 
    # friends (goout), grouped by the number of past class failures (failures | numeric 1-4)

    heatmap = data.groupby(['Walc', 'goout'])['G3'].mean().reset_index()

    fig4 = px.density_heatmap(heatmap, x = 'Walc', y = 'goout', z = 'G3', histfunc = 'avg',
                              title = 'Weekend alcholo consumption vs. going out')
    fig4.update_layout(autosize = True)
    

    return fig1, fig2, fig3, fig4

    


if __name__ == "__main__":
    app.run(debug=True, port=8070)
    

In [35]:
## ill also be doing any final edits on the actual .py file itself, so it might be slight different code than what's
## on here.

In [37]:
data['G3'].values

array([11, 11, 12, 14, 13, 13, 13, 13, 17, 13, 14, 13, 12, 13, 15, 17, 14,
       14,  7, 12, 14, 12, 14, 10, 10, 12, 12, 11, 13, 12, 11, 15, 15, 12,
       12, 11, 14, 13, 12, 12, 10, 11, 15, 10, 11, 11, 13, 17, 13, 12, 13,
       16,  9, 12, 13, 12, 15, 16, 14, 16, 16, 16, 10, 13, 12, 16, 12, 10,
       11, 15, 11, 10, 11, 14, 11, 11, 11, 13, 10, 11, 12,  9, 11, 13, 12,
       12, 11, 15, 11, 10, 11, 13, 12, 14, 12, 13, 11, 12, 13, 13,  8, 16,
       12, 10, 16, 10, 10, 14, 11, 14, 14, 11, 10, 18, 10, 14, 16, 15, 11,
       14, 14, 13, 13, 13, 11,  9, 11, 11, 15, 13, 12,  8, 11, 13, 12, 14,
       11, 11, 11, 15, 10, 13, 12, 11, 11, 10, 10, 14,  9, 11,  9, 13, 11,
       13, 11,  6, 12, 10, 11, 13, 11,  8, 11,  0, 10, 13, 11, 13,  8, 10,
       11, 11,  1, 10,  9,  8, 10,  8,  8,  8, 11, 18, 13, 17, 10, 18, 10,
       13, 15, 11, 14, 10, 11, 13, 11, 13, 17, 14, 16, 14, 11, 16, 14, 10,
       13, 12, 12, 10, 12, 16, 14, 12, 16, 11, 15, 12, 15, 13, 13,  8, 12,
       15, 13, 12, 12, 12